# 04 — Dataclasses, Properties, and Data Models

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/04', {1: 'dbf035d65fdef17d', 2: '275e46b329d52478', 3: '4a0d5e64ef02afa8', 4: '48a5ca3ce981dc0c', 5: '0052dec3cdeb4cf0', 6: '7ac8c047373a59df', 7: '5e721229584d779e', 8: '62548ccca89b62b1', 9: 'ed466611a25016fa'}, practice='intermediate_01_oop_and_collections')

## 1. @dataclass

- `@dataclass` writes `__init__`, `__repr__` and `__eq__` for you from annotated fields: `name: str`.
- Give a default after the field: `quantity: int = 0`. Fields with defaults must come after fields without.
- Never use a mutable default like `[]`; write `tags: list = field(default_factory=list)` so each object gets its own list.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Product:
    name: str
    price: float
    tags: list = field(default_factory=list)

p = Product("tea", 3.5)
print(p, p == Product("tea", 3.5))

**Exercise 1.** Create a `Product` for `"cake"` costing `4.0` and read its price.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

cake = Product(___, ___)
check(1, cake.price)

**Exercise 2.** Two dataclass objects with the same fields: are they `==`? Are they the same object?

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

a, b = Point(1, 2), Point(1, 2)
predicted = (___, ___)   # a == b, a is b
check(2, predicted)

**Exercise 3.** Two playlists must not share one list. Use `default_factory` and add a track to only one.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Playlist:
    name: str
    tracks: list = field(default_factory=___)

rock, jazz = Playlist("rock"), Playlist("jazz")
rock.tracks.append("song")
check(3, (rock.tracks, jazz.tracks))

## 2. Validation and frozen objects

- `__post_init__` runs after the generated `__init__`: check the fields there and raise `ValueError` for bad values.
- `@dataclass(frozen=True)` makes objects read-only (assigning raises `FrozenInstanceError`) and hashable, so they work in sets and as dict keys.
- `dataclasses.replace(obj, field=value)` makes a changed copy.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

home = Coordinate(51.5, -0.1)
print({home: "London"}, replace(home, lon=0.0))

**Exercise 4.** Reject a negative price in `__post_init__`. Record what happens for each price.

In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    def __post_init__(self):
        if ___:
            raise ValueError("price must not be negative")

results = []
for price in (3.0, -1.0):
    try:
        Product("x", price)
        results.append("ok")
    except ValueError:
        results.append("rejected")
check(4, results)

**Exercise 5.** Frozen coordinates are hashable. How many distinct points are in the set?

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: int
    y: int

points = {Point(1, 2), Point(1, 2), Point(2, 1)}
count = ___
check(5, count)

**Exercise 6.** Make a copy of the frozen point with `y` changed to 9.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Point:
    x: int
    y: int

moved = ___
check(6, (moved.x, moved.y))

## 3. Properties

- `@property` turns a method into a read-only attribute: `t.fahrenheit` with no parentheses.
- Add `@fahrenheit.setter` to allow assignment, converting or validating the value as it's stored.
- Store one source of truth (`_celsius`) and compute the rest.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32
    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5 / 9

t = Temperature(100)
print(t.fahrenheit)
t.fahrenheit = 32
print(t.celsius)

**Exercise 7.** Add a read-only `area` property to `Square`.

In [ ]:
class Square:
    def __init__(self, side):
        self.side = side
    @property
    def area(self):
        return ___

check(7, Square(5).area)

**Exercise 8.** Setting `fahrenheit` should update `celsius`. Set 212 °F and read the Celsius value.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32
    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = ___

t = Temperature(0)
t.fahrenheit = 212
check(8, t.celsius)

**Exercise 9.** A property without a setter can't be assigned. Which exception does `sq.area = 3` raise?

In [ ]:
class Square:
    def __init__(self, side): self.side = side
    @property
    def area(self): return self.side ** 2

try:
    Square(2).area = 3
except Exception as error:
    name = ___
check(9, name)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `Product`

A product that rejects a negative price or quantity.

In [ ]:
class Product:
    """A product line. Reject negative price or quantity in __post_init__.

    `total_value` is a property: price * quantity.
    """
    sku: str
    price: float
    quantity: int = 0

    def __post_init__(self):
        if ___:
            raise ValueError("price and quantity must be nonnegative")

    @property
    def total_value(self) -> float:
        return ___

In [ ]:
check('Product')

### 2. `Playlist`

A playlist that owns its own list of tracks.

In [ ]:
class Playlist:
    """A playlist that owns its own list of tracks. Hint: field(default_factory=list)."""
    name: str
    tracks: list[str] = field(default_factory=list)

    def add(self, track: str) -> None:
        self.tracks.append(___)

In [ ]:
check('Playlist')

### 3. `Coordinate`

An immutable, hashable point.

In [ ]:
class Coordinate:
    """An immutable, hashable point. Hint: @dataclass(frozen=True)."""
    x: float
    y: float

In [ ]:
check('Coordinate')

### Independent build — `Temperature`

Temperature(celsius) exposes writable celsius and fahrenheit properties. Both refer to the same temperature (0 °C = 32 °F, a change of 5 °C = 9 °F). Reject values below −273.15 °C with ValueError, including during construction.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
class Temperature:
    pass  # Implement the public interface in the ticket.

In [ ]:
check('Temperature')

### 5. `Inventory`

Hold products by SKU; reject adding the same SKU twice.

In [ ]:
class Inventory:
    """Holds Product objects keyed by SKU; adding an existing SKU raises ValueError."""

    def __init__(self):
        self._products: dict[str, Product] = {}

    def add(self, product: Product) -> None:
        if ___:
            raise ValueError(f"duplicate sku: {product.sku}")
        self._products[product.sku] = product

    def total_value(self) -> float:
        return ___

    def __len__(self) -> int:
        return ___

In [ ]:
check('Inventory')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why use field(default_factory=list)?

   - **a)** So each instance gets its own list
   - **b)** It's faster
   - **c)** Lists aren't allowed in dataclasses

**2.** Where do you validate dataclass fields?

   - **a)** __init__ (written by hand)
   - **b)** __repr__
   - **c)** __post_init__

**3.** What does frozen=True give you?

   - **a)** Immutable, hashable instances
   - **b)** Automatic validation
   - **c)** Faster attribute access

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/04', {1: '2a28d7884674', 2: 'f5bdfec142a3', 3: '7e19a27eef80'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [05 — Iterables, Iterators, and Protocols](05_iterables_and_iterators.ipynb)